# Options Problem

As covered with our options pricing framework, we learned that we can generate the fair price of an option contract if we input volatility. But, there is one thing to note here. This framework specifically assumes that volatility is constant: it takes one input and then spits out an output. But volatility is rarely every this static. And so, we look to creating what is called an implied volatility surface, which is a 3D topographical map of the fair value of an option contract, where the x-axis is the possible strike prices $K$, and the z-axis is the possible TTMs $\tau$. 

## Heston Framework

We will specifically be looking at the volatility surface created through Heston Stochastic Volatility. First, consider the generic SDE that is Geometric Brownian Motion of variance $v_t$ in general Itô form:
$$dv_t = \mu(v_t, t)dt + \sigma(v_t, t)dW_t^v$$
Here, $\mu(v_t,t)$ represents the drift, and $\sigma(v_t, t)$ represents the stochastic part. This heston model must choose functional terms to represent the drift and randomness terms, respectively. For drift, we specifically use Ornstein-Uhlenbeck linear mean-reverting drift:
$$\mu(v_t) = \kappa(\theta - v_t)$$
Here, $\theta$ is the long-term variance, and $\kappa$ is the rate of mean reversion, or how quickly the variance pulls back to $\theta$. For the random term, we must uphold this condition: $v_t=\sigma_t^2$, meaning variance can never be negative. If we scale linearly with volatility, we may produce negative numbers. So, we choose our random term to be proportional to the squareroot of the current variance:
$$\sigma(v_t) = \xi \sqrt{v_t}$$
Here, $\xi$ is specifically the volatility-of-volatility. This ensures that as $v_t$ approaches zero, the random term disappears, and the drift term remains positive. So, the complete Heston variance SDE is written as:
$$dv_t = \kappa(\theta - v_t)dt + \xi \sqrt{v_t} dW_t^v$$
To ensure that the variance is never zero, the Feller condition is established. It states that the rate at drift pulls away from zero must outweigh the volatility trying to push it into zero:
$$2\kappa\theta > \xi^2$$

## Deriving Heston Parameters Naively

The parameters of the Heston model, rate of mean reversion $\kappa$, long-term variance $\theta$, vol-of-vol $\xi$, and correlation $\rho$ can be directly solved for using statistical interference. But note that this method isn't typically used in practice, as it calibrates the model to historical behavior. The more robust method is covered in the next section. First, discretize the SDE so that it is in a computable form:
$$v_{t+\Delta t} - v_t = \kappa(\theta - v_t)\Delta t + \xi \sqrt{v_t \Delta t} \epsilon_{t+1}$$
Here, $v_t$ is the current variance $v_{t+\Delta t}$ is the next period's variance, $\Delta t$ is a time step (typically one day), and $\epsilon_{t+1}\sim\mathcal{N}(0,1)$. Note that we usually have to scale regression residuals by dividing by $\sqrt{v_t\Delta t}$ to eradicate heteroskedasticity. We use a variety of optimization means to determine the most optimal $\kappa$ and $\theta$ values (LM is the most preferred), but note $\xi$ always correlates to the variance of the residual error terms. 

## Kolmogorov Backwards Equation

Now let's derive the framework for which our characteristic function must lie on top of. Start with the risk-neutral asset pricing equation, and our variance equation: 
$$dx_t = \left(r - \frac{1}{2}v_t\right)dt + \sqrt{v_t} dW_t^{(1)}$$
$$dv_t = \kappa(\theta - v_t)dt + \xi \sqrt{v_t} dW_t^{(2)}$$
We must also have the correlation $\rho$ between the asset shocks and the variance shocks. Now let's look at a property of Martingales (the expected value of the next step is equal to the current value). A characteristic function $\phi(x,v,t)$ is defined as the conditional expected value of an event under a risk neutral framework $\mathbb{E}$:
$$\phi(x_t, v_t, t) = \mathbb{E}^{\mathbb{Q}}_t [e^{i u x_T}]$$
The law of iterated expectation states that characteristic functions are martingales. Thus, it must have an expected drift of zero. When we expand out $d\phi(x_t+dx_t, v_t+dv_t, t+dt)$ around $(x_t, v_t, t)$, we get the following Taylor polynomial:
$$d\phi = \frac{\partial \phi}{\partial t}dt + \frac{\partial \phi}{\partial x}dx_t + \frac{\partial \phi}{\partial v}dv_t + \frac{1}{2}\frac{\partial^2 \phi}{\partial x^2}(dx_t)^2 + \frac{1}{2}\frac{\partial^2 \phi}{\partial v^2}(dv_t)^2 + \frac{\partial^2 \phi}{\partial x \partial v}(dx_t)(dv_t)$$
Now we must apply Itô's Lemma to this equation. For $(dx_t)^2$, all $d_t$ terms must drop out, so we are left with $( \sqrt{v} dW^S )^2 = v dt$. For $(dv_t)^2$, we have a similar pattern, and are left with $( \xi \sqrt{v} dW^{v} )^2 = \xi^2 v dt$. Finally, for $(dv_t)(dx_t)$, we have $(\sqrt{v} dW^{S})(\xi \sqrt{v} dW^{e}) = \xi v (dW^{S} dW^{e}) = \rho \xi v dt$. Notice that since we multiplied two Brownian motions of different randomness measure, we have a residual correlation $\rho$ in addition to $dt$. Note that since every term is in terms of $dt$, this PDE inherently measures drift:
$$\text{Drift} = \left[ \frac{\partial \phi}{\partial t} + \left(r - \frac{1}{2}v\right)\frac{\partial \phi}{\partial x} + \kappa(\theta - v)\frac{\partial \phi}{\partial v} + \frac{1}{2}v\frac{\partial^2 \phi}{\partial x^2} + \frac{1}{2}\xi^2 v \frac{\partial^2 \phi}{\partial v^2} + \rho \xi v \frac{\partial^2 \phi}{\partial x \partial v} \right] dt$$
And since we established that our characteristic function which abides by the parameters of the Taylor polynomial is a martingale and has an expected drift of zero, that means the final PDE is:
$$\frac{\partial \phi}{\partial t} + \left(r - \frac{1}{2}v\right)\frac{\partial \phi}{\partial x} + \kappa(\theta - v)\frac{\partial \phi}{\partial v} + \frac{1}{2}v\frac{\partial^2 \phi}{\partial x^2} + \frac{1}{2}\xi^2 v \frac{\partial^2 \phi}{\partial v^2} + \rho \xi v \frac{\partial^2 \phi}{\partial x \partial v} = 0$$
Let's analyze this PDE:
- **The Drift Terms ($\frac{\partial \phi}{\partial v}$):** This acknowledges that this process is mean reverting, and tells the Heston engine to expect the volatility to move towards the long-term average $\theta$.
- **The Diffusion Terms ($\frac{\partial^2 \phi}{\partial x^2}$):** This acknowledges that there is residual variance that must be accounted for, leading the stock price and volatility to jump around. 
- **The Correlation Term ($\frac{\partial^2 \phi}{\partial x \partial v}$):** This lets the engine know that asset shocks and volatility shocks are heavily correlated, and that if one spikes, so will the other. 

## Heston Characteristic Function

A characteristic function is used to translate random variables into the complex domain. The Heston framework revolves around using a Fast Fourier Transform to compute possible fair option values, so a characteristic function is necessary here. In the following derivation, $x=\ln(S_t)$. In our case, we want to consider the characteristic function as $\phi(x, v, t; u) = \mathbb{E}^{\mathbb{Q}}_t[e^{i u x_T}]$.  What this effectively lets us do, is consider $x$ in complex space at the terminal time $T$. This characteristic function must specifically satisfy the above Kolmogorov Backwards PDE. In order to solve this PDE, Steve Heston proposed an affine guess, to assume the solution takes an exponential form to separate variables $x$ and $v$:
$$\phi(x, v, t; u) = \exp \left( C(\tau) + D(\tau)v + i u x \right)$$
Here, $C(\tau)$ and $D(\tau)$ are both functions of time $t$. Taking all derivatives of this imaginary function yields:
- $\frac{\partial \phi}{\partial t} = \left( -C'(\tau) - D'(\tau)v \right) \phi$
- $\frac{\partial \phi}{\partial x} = i u \phi$
- $\frac{\partial^2 \phi}{\partial x^2} = (i u)^2 \phi = -u^2 \phi$
- $\frac{\partial \phi}{\partial v} = D(\tau) \phi$
- $\frac{\partial^2 \phi}{\partial v^2} = D(\tau)^2 \phi$
- $\frac{\partial^2 \phi}{\partial x \partial v} = i u D(\tau) \phi$
Re-arranging these back into the PDE and then grouping terms multiplied by $v$ vs. those not multiplied by $v$ yields:
$$v \left[ -D'(\tau) - \frac{1}{2}i u - \frac{1}{2}u^2 - \kappa D(\tau) + \frac{1}{2}\xi^2 D(\tau)^2 + \rho \xi i u D(\tau) \right] + \left[ -C'(\tau) + r i u + \kappa \theta D(\tau) \right] = 0$$
In order for the equation to be true, both sides must equal zero, leasing us with two differential equations:
$$C'(\tau) = r i u + \kappa \theta D(\tau)$$
$$D'(\tau) = \frac{1}{2}\xi^2 D(\tau)^2 + (\rho \xi i u - \kappa) D(\tau) - \frac{1}{2}(u^2 + i u)$$
In order to solve the second equation, it's necessary to define three constants:
- $a = -\frac{1}{2}(u^2 + i u)$
- $b = \kappa - \rho \xi i u$
- $c = \frac{1}{2}\xi^2$
This now yields an equation in the form $D' = a - bD + cD^2$. We will omit the algebra here for brevity's sake, but understand that we effectively rewrite this equation in terms of the roots, use partial-fraction decompisition in order to integrate, and then use more algebra to solve for $D$:
$$D(\tau) = \frac{b - d}{\xi^2} \left( \frac{1 - e^{-d \tau}}{1 - g e^{-d \tau}} \right)$$
Then, in order to solve for $C$, we plug $D$ into this equation, and then use more algebra and integration to yield:
$$C(\tau) = r i u \tau + \frac{\kappa \theta}{\xi^2} \left[ (b - d)\tau - 2 \ln \left( \frac{1 - g e^{-d \tau}}{1 - g} \right) \right]$$
Our entire Heston Characteristic Function is now $\phi(u) = \exp(C(\tau) + D(\tau)v + i u x)$. And so, we can now put our aggregate our random variables $x_t$ and $v_t$ into complex space in order to run the FFT. 

## Carr-Madan Transformation

We'll assume you have a prior understanding as to what a Fourier Transform is. Consider that we want to price a European call option with a strike price $K$. Here, $k=\ln(K)$ and $s=\ln(S)$. In continuous form, this is expressed as:
$$C_T(k) = e^{-rT} \int_{k}^{\infty} (e^{s} - e^k) q(s) ds$$
But note that this is not natively square-integrable, meaning the FFT would not be able to work (the function does not take up finite space). So, we typically multiply by a dampening factor:
$$c(k) = e^{\alpha k} C_T(k)$$
Now we can actually take the Fourier transform of $C_T(k)$:
$$\psi(\nu) = \int_{-\infty}^{\infty} e^{i \nu k} e^{\alpha k} C_T(k) dk$$
If we substitute the integral definition of $C_T(k)$ back into this, we get:
$$\psi(\nu) = \int_{-\infty}^{\infty} e^{(\alpha + i \nu) k} \left[ e^{-rT} \int_{k}^{\infty} (e^s - e^k) q(s) ds \right] dk$$
We can then swap our order of integration:
$$\psi(\nu) = e^{-rT} \int_{-\infty}^{\infty} q(s) \left[ \int_{-\infty}^{s} e^{(\alpha + i \nu) k} (e^s - e^k) dk \right] ds$$
Now we evaluate the inner integral:
$$\int_{-\infty}^{s} \left( e^{s + (\alpha + i \nu)k} - e^{(1 + \alpha + i \nu)k} \right) dk$$$$= \left[ \frac{e^{s + (\alpha + i \nu)k}}{\alpha + i \nu} - \frac{e^{(1 + \alpha + i \nu)k}}{1 + \alpha + i \nu} \right]_{-\infty}^{s}$$
$$= e^{(1 + \alpha + i \nu)s} \left( \frac{1}{\alpha + i \nu} - \frac{1}{1 + \alpha + i \nu} \right)$$
$$= \frac{e^{i (\nu - (\alpha + 1)i) s}}{(\alpha + i \nu)(1 + \alpha + i \nu)}$$
Substituting this back into the main integral, we find:
$$\psi(\nu) = e^{-rT} \int_{-\infty}^{\infty} q(s) \frac{e^{i (\nu - (\alpha + 1)i) s}}{\alpha^2 + \alpha - \nu^2 + i(2\alpha + 1)\nu} ds$$
Since the numerator is simply the expectation of the $e^{i (\nu - (\alpha + 1)i) s}$, the numerator is our characteristic function evaluated at frequency $u=\nu - (\alpha + 1)i$. Therefore, our Fourier transform is expressed as:
$$\psi(\nu) = \frac{e^{-rT} \phi(\nu - (\alpha + 1)i)}{\alpha^2 + \alpha - \nu^2 + i(2\alpha + 1)\nu}$$
Note that expectation inherently discretizes the integral. Now, this only produces a frequency. In order to recover the actuall call price of the option contract, we must run the inverse FFT:
$$C_T(k) = \frac{e^{-\alpha k}}{\pi} \int_{0}^{\infty} e^{-i \nu k} \psi(\nu) d\nu$$

## Discretizing the FFT

Note that we cannot compute something continuous. Therefore, we must express the inverse FFT in the form of summations.
- Define a set of $\nu$ values. For some large $N$ (either $2048$ or $4096$), we define every $\nu_j=\eta(j-1)$ where $\eta$ is the step size.
- Define a grid for the log-strike prices separated by step size $\lambda$, starting from a deep OTM strike $-b$ (we set it to negative because it's OTM).
The fully discretized FFT is then:
$$C_T(k_u) \approx \frac{e^{-\alpha k_u}}{\pi} \sum_{j=1}^{N} e^{-i \nu_j k_u} \psi(\nu_j) w_j \eta$$
Then after re-arranging the exponents:
$$e^{-i \nu_j k_u} = e^{-i \eta(j - 1) (-b + \lambda(u - 1))}$$
$$= e^{i b \nu_j} e^{-i \lambda \eta (j - 1)(u - 1)}$$
And also consider the native boundary constraint for FFTs:
$$\lambda \eta = \frac{2\pi}{N}$$
The final, discretized FFT for Heston Stochastic Volatility is:
$$C_T(k_u) \approx \frac{e^{-\alpha k_u}}{\pi} \sum_{j=1}^{N} \left[ e^{i b \nu_j} \psi(\nu_j) w_j \eta \right] e^{-i \frac{2\pi}{N} (j - 1)(u - 1)}$$